# EWOC Health Checks V6.1 — Clean Notebook

This notebook reads the collector bundle and produces three compact results:

1. health summary;
2. detected signals;
3. key presentation metrics.

It keeps the production data, prediction, deployment, pipeline, and trusted
evaluation checks. The only optional challenger is a lightweight TF-IDF +
logistic-regression model. No transformer, sentence-embedding, or zero-shot
language model is loaded.


## 1. Core configuration and small helpers


In [ ]:
from __future__ import annotations

import json
import math
import re
from collections import Counter
from dataclasses import asdict, dataclass, fields
from datetime import datetime, timedelta, timezone
from pathlib import Path
from typing import Any, Mapping

import numpy as np
import pandas as pd
from IPython.display import display

PASS, WARN, FAIL, UNKNOWN = "PASS", "WARN", "FAIL", "UNKNOWN"
TOKEN_RE = re.compile(r"[A-Za-z0-9_]+")


@dataclass
class HealthConfig:
    challenger: str = "none"              # "none" or "tfidf"
    label_maturity_hours: int = 168
    evaluation_max_rows: int = 5_000
    labels_independent_of_model: bool | None = None
    production_training_cutoff: str | None = None
    as_of: str | None = None
    output: str | None = "ewoc_agent_evidence.json"


@dataclass(frozen=True)
class Thresholds:
    min_tickets: int = 50
    freshness_warn_hours: float = 48
    freshness_fail_hours: float = 168
    volume_change_warn_pct: float = 50
    volume_change_fail_pct: float = 80
    missing_text_warn_pct: float = 1
    missing_text_fail_pct: float = 5
    short_text_chars: int = 20
    short_text_warn_pct: float = 10
    short_text_fail_pct: float = 25
    duplicate_warn_pct: float = 0.5
    duplicate_fail_pct: float = 2
    average_confidence_warn: float = 0.65
    average_confidence_fail: float = 0.50
    low_confidence_value: float = 0.60
    low_confidence_warn_pct: float = 20
    low_confidence_fail_pct: float = 40
    prediction_jsd_warn: float = 0.10
    prediction_jsd_fail: float = 0.20
    unseen_token_warn_pct: float = 15
    unseen_token_fail_pct: float = 30
    token_jsd_warn: float = 0.10
    token_jsd_fail: float = 0.20
    accuracy_warn: float = 0.70
    accuracy_fail: float = 0.65
    weighted_f1_warn: float = 0.70
    weighted_f1_fail: float = 0.55
    api_success_warn_pct: float = 99
    api_success_fail_pct: float = 95
    api_latency_warn_ms: float = 1_000
    api_latency_fail_ms: float = 3_000
    challenger_disagreement_warn_pct: float = 15
    challenger_disagreement_fail_pct: float = 30


ALIASES = {
    "id": ("EWOC_ID", "TICKET_ID", "ID"),
    "text": ("DESCRIPTION", "TICKET_TEXT", "TEXT"),
    "actual": ("_ACTUAL_LABEL", "ACTUAL_LABEL", "TYPE_WO", "Y_TRUE"),
    "prediction": ("_PREDICTION_LABEL", "PREDICTION_LABEL", "PREDICTED_LABEL", "Y_PRED"),
    "confidence": ("_PREDICTION_CONFIDENCE", "PREDICTION_CONFIDENCE", "CONFIDENCE"),
    "failed": ("_PREPROCESSING_FAILED", "PREPROCESSING_FAILED", "INFERENCE_ERROR"),
    "api_ok": ("_PREDICTION_API_OK", "API_PREDICTION_OK"),
    "api_latency": ("_PREDICTION_API_LATENCY_MS", "API_LATENCY_MS"),
    "created": ("CREATED_DATE", "CREATED_AT"),
    "updated": ("UPDATE_DATE", "UPDATED_AT", "LAST_UPDATED"),
}


def dataclass_from(cls, values=None):
    values = dict(values or {})
    allowed = {item.name for item in fields(cls)}
    return cls(**{key: value for key, value in values.items() if key in allowed})


def column(frame: pd.DataFrame | None, logical: str) -> str | None:
    if frame is None:
        return None
    names = {str(name).upper(): str(name) for name in frame.columns}
    return next((names[name] for name in ALIASES[logical] if name in names), None)


def percent(count: float, total: int) -> float:
    return round(100 * float(count) / total, 4) if total else 0.0


def high_is_bad(value: float, warn: float, fail: float) -> str:
    return FAIL if value >= fail else WARN if value >= warn else PASS


def low_is_bad(value: float, warn: float, fail: float) -> str:
    return FAIL if value <= fail else WARN if value <= warn else PASS


def check(value: Any, status: str, description: str, *, source=None, threshold=None) -> dict:
    result = {"value": value, "status": status, "description": description}
    if source:
        result["source"] = source
    if threshold is not None:
        result["threshold"] = threshold
    return result


def unknown(description: str) -> dict:
    return check(None, UNKNOWN, description)


def distribution(values: pd.Series) -> dict[str, float]:
    clean = values.dropna().astype(str).str.strip().str.casefold()
    clean = clean[clean.ne("")]
    return clean.value_counts(normalize=True).to_dict()


def js_divergence(left: Mapping[str, float], right: Mapping[str, float]) -> float | None:
    keys = sorted(set(left) | set(right))
    if not keys:
        return None
    p = np.array([left.get(key, 0.0) for key in keys], dtype=float)
    q = np.array([right.get(key, 0.0) for key in keys], dtype=float)
    if p.sum() == 0 or q.sum() == 0:
        return None
    p, q = p / p.sum(), q / q.sum()
    middle = (p + q) / 2
    kl = lambda x: np.sum(np.where(x > 0, x * np.log2(x / middle), 0.0))
    return float((kl(p) + kl(q)) / 2)


def token_counts(values: pd.Series, limit: int = 500_000) -> Counter:
    result = Counter()
    for value in values.dropna():
        result.update(TOKEN_RE.findall(str(value).casefold()))
        if result.total() >= limit:
            break
    return result


def parse_time(value: Any) -> datetime | None:
    if value in (None, ""):
        return None
    stamp = pd.Timestamp(value)
    return stamp.tz_localize("UTC").to_pydatetime() if stamp.tz is None else stamp.tz_convert("UTC").to_pydatetime()


def json_safe(value: Any) -> Any:
    if isinstance(value, dict):
        return {str(key): json_safe(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(item) for item in value]
    if isinstance(value, (datetime, pd.Timestamp)):
        return value.isoformat()
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, float) and not math.isfinite(value):
        return None
    return value


## 2. Load the collector bundle and evaluate trusted new data


In [ ]:
def load_collection(directory: str | Path) -> dict[str, Any]:
    root = Path(directory)
    manifest = json.loads((root / "collection_manifest.json").read_text(encoding="utf-8"))
    bundle = {
        name: pd.read_pickle(root / item["path"])
        for name, item in manifest.get("datasets", {}).items()
    }
    bundle.update({
        "model_metadata": manifest.get("model_metadata", {}),
        "source_metadata": manifest.get("source_metadata", {}),
        "pipeline_telemetry": manifest.get("pipeline_telemetry", {}),
        "api_test_result": manifest.get("api_test_result", {}),
        "labels_independent_of_model": bool(manifest.get("labels_independent_of_model", False)),
        "collector_config": manifest.get("collection_config", manifest.get("collector_config", {})),
    })
    return bundle


def classification_scores(actual: pd.Series, predicted: pd.Series) -> dict[str, Any]:
    from sklearn.metrics import accuracy_score, f1_score

    pairs = pd.DataFrame({"actual": actual, "predicted": predicted}).dropna()
    pairs = pairs[
        pairs["actual"].astype(str).str.strip().ne("")
        & pairs["predicted"].astype(str).str.strip().ne("")
    ]
    if pairs.empty:
        return {"count": 0, "accuracy": None, "weighted_f1": None}
    actual_clean = pairs["actual"].astype(str).str.strip().str.casefold()
    pred_clean = pairs["predicted"].astype(str).str.strip().str.casefold()
    return {
        "count": len(pairs),
        "accuracy": float(accuracy_score(actual_clean, pred_clean)),
        "weighted_f1": float(f1_score(actual_clean, pred_clean, average="weighted", zero_division=0)),
    }


def tfidf_challenger(training: pd.DataFrame, evaluation: pd.DataFrame) -> dict[str, Any]:
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.linear_model import LogisticRegression
    from sklearn.pipeline import Pipeline

    train_text, train_label = column(training, "text"), column(training, "actual")
    eval_text, eval_pred = column(evaluation, "text"), column(evaluation, "prediction")
    if not all((train_text, train_label, eval_text, eval_pred)):
        return {"available": False, "reason": "required columns are missing"}

    train = training[[train_text, train_label]].dropna().copy()
    train = train[train[train_text].astype(str).str.strip().ne("")]
    counts = train[train_label].astype(str).value_counts()
    train = train[train[train_label].astype(str).isin(counts[counts >= 2].index)]
    if len(train) < 100 or train[train_label].nunique() < 2 or evaluation.empty:
        return {"available": False, "reason": "not enough training rows or classes"}

    model = Pipeline([
        ("tfidf", TfidfVectorizer(
            ngram_range=(1, 2), sublinear_tf=True, min_df=2,
            max_features=30_000, stop_words="english",
        )),
        ("classifier", LogisticRegression(
            class_weight="balanced", max_iter=500, solver="liblinear", random_state=42,
        )),
    ])
    model.fit(train[train_text].fillna("").astype(str), train[train_label].astype(str))
    challenger = pd.Series(
        model.predict(evaluation[eval_text].fillna("").astype(str)), index=evaluation.index
    )
    production = evaluation[eval_pred].fillna("").astype(str)
    disagreement = percent(
        challenger.str.casefold().ne(production.str.casefold()).sum(), len(evaluation)
    )
    return {
        "available": True,
        "model": "tfidf_logistic_regression",
        "training_rows": len(train),
        "disagreement_rate_pct": disagreement,
    }


def evaluate_independent(bundle: Mapping[str, Any], config: HealthConfig) -> dict[str, Any]:
    evaluation = bundle.get("evaluation", pd.DataFrame()).copy()
    actual_col, pred_col = column(evaluation, "actual"), column(evaluation, "prediction")
    if evaluation.empty or not actual_col or not pred_col:
        return {"available": False, "trusted": False, "reason": "evaluation labels or predictions are missing"}

    text_col = column(evaluation, "text")
    keep = evaluation[actual_col].notna() & evaluation[pred_col].notna()
    if text_col:
        keep &= evaluation[text_col].fillna("").astype(str).str.strip().ne("")
    evaluation = evaluation.loc[keep].copy()

    windows = bundle.get("source_metadata", {}).get("windows", {})
    cutoff = parse_time(
        config.production_training_cutoff
        or windows.get("evaluation", {}).get("start")
        or windows.get("challenger_training", {}).get("end")
    )
    as_of = parse_time(config.as_of) or datetime.now(timezone.utc)
    date_col = column(evaluation, "updated") or column(evaluation, "created")
    if date_col:
        dates = pd.to_datetime(evaluation[date_col], errors="coerce", utc=True)
        mask = dates.notna() & dates.le(as_of - timedelta(hours=config.label_maturity_hours))
        if cutoff:
            mask &= dates.gt(cutoff)
        evaluation = evaluation.loc[mask].copy()

    id_col = column(evaluation, "id")
    if id_col:
        evaluation = evaluation.drop_duplicates(id_col, keep="last")
    if len(evaluation) > config.evaluation_max_rows:
        evaluation = evaluation.sample(config.evaluation_max_rows, random_state=42)

    scores = classification_scores(evaluation[actual_col], evaluation[pred_col])
    trusted_flag = (
        bool(bundle.get("labels_independent_of_model", False))
        if config.labels_independent_of_model is None
        else bool(config.labels_independent_of_model)
    )
    trusted = bool(trusted_flag and cutoff is not None and scores["count"])
    result = {
        "available": bool(scores["count"]),
        "trusted": trusted,
        "count": scores["count"],
        "accuracy": scores["accuracy"] if trusted else None,
        "weighted_f1": scores["weighted_f1"] if trusted else None,
        "cutoff": cutoff,
        "label_maturity_hours": config.label_maturity_hours,
    }

    if config.challenger.casefold() == "tfidf":
        training = bundle.get("challenger_training", pd.DataFrame())
        result["challenger"] = tfidf_challenger(training, evaluation)
    else:
        result["challenger"] = {"available": False, "reason": "disabled"}
    return result


## 3. Essential health evaluation


In [ ]:
def evaluate_health(
    bundle: Mapping[str, Any],
    config: HealthConfig | Mapping[str, Any] | None = None,
    thresholds: Thresholds | Mapping[str, Any] | None = None,
) -> dict[str, Any]:
    cfg = config if isinstance(config, HealthConfig) else dataclass_from(HealthConfig, config)
    t = thresholds if isinstance(thresholds, Thresholds) else dataclass_from(Thresholds, thresholds)
    current = bundle.get("current", pd.DataFrame()).copy()
    reference = bundle.get("reference", pd.DataFrame()).copy()
    if not isinstance(current, pd.DataFrame):
        raise TypeError("bundle['current'] must be a pandas DataFrame")

    independent = evaluate_independent(bundle, cfg)
    n, n_reference = len(current), len(reference)
    input_health = {
        "ticket_count": check(
            n, PASS if n >= t.min_tickets else FAIL,
            "Current Oracle ticket count.", threshold={"minimum": t.min_tickets},
        )
    }

    if n_reference:
        change = 100 * (n - n_reference) / n_reference
        input_health["ticket_volume_change_pct"] = check(
            round(change, 4), high_is_bad(abs(change), t.volume_change_warn_pct, t.volume_change_fail_pct),
            "Current ticket-volume change versus reference.",
            threshold={"warn_abs_pct": t.volume_change_warn_pct, "fail_abs_pct": t.volume_change_fail_pct},
        )
    else:
        input_health["ticket_volume_change_pct"] = unknown("Reference ticket count is unavailable.")

    date_col = column(current, "updated") or column(current, "created")
    if date_col and n:
        latest = pd.to_datetime(current[date_col], errors="coerce", utc=True).max()
        hours = (pd.Timestamp.now(tz="UTC") - latest).total_seconds() / 3600 if pd.notna(latest) else None
        input_health["source_freshness_hours"] = (
            check(
                round(hours, 3), high_is_bad(hours, t.freshness_warn_hours, t.freshness_fail_hours),
                "Age of the latest Oracle work order.", source=date_col,
                threshold={"warn_hours": t.freshness_warn_hours, "fail_hours": t.freshness_fail_hours},
            ) if hours is not None else unknown("Latest Oracle timestamp is unavailable.")
        )
    else:
        input_health["source_freshness_hours"] = unknown("Oracle update timestamp is unavailable.")

    text_col = column(current, "text")
    if text_col:
        text = current[text_col]
        clean_text = text.fillna("").astype(str).str.strip()
        missing_rate = percent(text.isna().sum() + (text.notna() & clean_text.eq("")).sum(), n)
        short_rate = percent(clean_text.ne("").mul(clean_text.str.len().lt(t.short_text_chars)).sum(), n)
        input_health["missing_text_rate_pct"] = check(
            missing_rate, high_is_bad(missing_rate, t.missing_text_warn_pct, t.missing_text_fail_pct),
            "Missing or empty work-order descriptions.", source=text_col,
            threshold={"warn_pct": t.missing_text_warn_pct, "fail_pct": t.missing_text_fail_pct},
        )
        input_health["short_text_rate_pct"] = check(
            short_rate, high_is_bad(short_rate, t.short_text_warn_pct, t.short_text_fail_pct),
            f"Descriptions shorter than {t.short_text_chars} characters.", source=text_col,
            threshold={"warn_pct": t.short_text_warn_pct, "fail_pct": t.short_text_fail_pct},
        )
    else:
        input_health["missing_text_rate_pct"] = unknown("Description column is unavailable.")
        input_health["short_text_rate_pct"] = unknown("Description column is unavailable.")

    id_col = column(current, "id")
    if id_col:
        duplicate_rate = percent(current[id_col].notna().sum() - current[id_col].dropna().nunique(), n)
        input_health["duplicate_ticket_id_rate_pct"] = check(
            duplicate_rate, high_is_bad(duplicate_rate, t.duplicate_warn_pct, t.duplicate_fail_pct),
            "Repeated ticket IDs.", source=id_col,
            threshold={"warn_pct": t.duplicate_warn_pct, "fail_pct": t.duplicate_fail_pct},
        )
    else:
        input_health["duplicate_ticket_id_rate_pct"] = unknown("Ticket ID column is unavailable.")

    failed_col = column(current, "failed")
    if failed_col:
        failed = current[failed_col].fillna(False).astype(str).str.casefold().isin({"true", "1", "yes", "failed", "error"})
        failure_rate = percent(failed.sum(), n)
        input_health["preprocessing_failure_rate_pct"] = check(
            failure_rate, high_is_bad(failure_rate, 1, 5),
            "Rows failing preprocessing or inference.", source=failed_col,
        )
    else:
        input_health["preprocessing_failure_rate_pct"] = unknown("Preprocessing failure flag is unavailable.")

    comparison = {}
    reference_text = column(reference, "text")
    if text_col and reference_text and n_reference:
        current_tokens = token_counts(current[text_col])
        reference_tokens = token_counts(reference[reference_text])
        unseen = percent(sum(count for token, count in current_tokens.items() if token not in reference_tokens), current_tokens.total())
        token_shift = js_divergence(current_tokens, reference_tokens)
        comparison["unseen_token_rate_pct"] = check(
            unseen, high_is_bad(unseen, t.unseen_token_warn_pct, t.unseen_token_fail_pct),
            "Current tokens absent from reference text.",
            threshold={"warn_pct": t.unseen_token_warn_pct, "fail_pct": t.unseen_token_fail_pct},
        )
        comparison["token_distribution_jsd"] = (
            check(
                round(token_shift, 6), high_is_bad(token_shift, t.token_jsd_warn, t.token_jsd_fail),
                "Current/reference token-distribution divergence.",
                threshold={"warn": t.token_jsd_warn, "fail": t.token_jsd_fail},
            ) if token_shift is not None else unknown("Token-distribution divergence is unavailable.")
        )
    else:
        comparison["unseen_token_rate_pct"] = unknown("Reference text is unavailable.")
        comparison["token_distribution_jsd"] = unknown("Reference text is unavailable.")

    prediction_health = {}
    pred_col, ref_pred_col = column(current, "prediction"), column(reference, "prediction")
    if pred_col:
        present = current[pred_col].fillna("").astype(str).str.strip().ne("")
        coverage = percent(present.sum(), n)
        prediction_health["prediction_coverage_pct"] = check(
            coverage, PASS if coverage >= 99 else WARN if coverage >= 95 else FAIL,
            "Rows with production predictions.", source=pred_col,
        )
        if ref_pred_col and n_reference:
            pred_shift = js_divergence(distribution(current[pred_col]), distribution(reference[ref_pred_col]))
            comparison["prediction_distribution_jsd"] = (
                check(
                    round(pred_shift, 6), high_is_bad(pred_shift, t.prediction_jsd_warn, t.prediction_jsd_fail),
                    "Current/reference prediction-distribution divergence.",
                    threshold={"warn": t.prediction_jsd_warn, "fail": t.prediction_jsd_fail},
                ) if pred_shift is not None else unknown("Prediction-distribution divergence is unavailable.")
            )
        else:
            comparison["prediction_distribution_jsd"] = unknown("Reference predictions are unavailable.")
    else:
        prediction_health["prediction_coverage_pct"] = unknown("Production predictions are unavailable.")
        comparison["prediction_distribution_jsd"] = unknown("Production predictions are unavailable.")

    confidence_col = column(current, "confidence")
    if confidence_col:
        confidence = pd.to_numeric(current[confidence_col], errors="coerce").dropna().clip(0, 1)
        if len(confidence):
            average = float(confidence.mean())
            low_rate = percent(confidence.lt(t.low_confidence_value).sum(), len(confidence))
            prediction_health["average_prediction_confidence"] = check(
                round(average, 6), low_is_bad(average, t.average_confidence_warn, t.average_confidence_fail),
                "Mean maximum class probability.", source=confidence_col,
                threshold={"warn_below": t.average_confidence_warn, "fail_below": t.average_confidence_fail},
            )
            prediction_health["low_confidence_rate_pct"] = check(
                low_rate, high_is_bad(low_rate, t.low_confidence_warn_pct, t.low_confidence_fail_pct),
                f"Predictions below {t.low_confidence_value:.2f} confidence.", source=confidence_col,
                threshold={"warn_pct": t.low_confidence_warn_pct, "fail_pct": t.low_confidence_fail_pct},
            )
        else:
            prediction_health["average_prediction_confidence"] = unknown("Prediction confidence is unavailable.")
            prediction_health["low_confidence_rate_pct"] = unknown("Prediction confidence is unavailable.")
    else:
        prediction_health["average_prediction_confidence"] = unknown("Prediction confidence is unavailable.")
        prediction_health["low_confidence_rate_pct"] = unknown("Prediction confidence is unavailable.")

    api_ok_col, api_latency_col = column(current, "api_ok"), column(current, "api_latency")
    if api_ok_col:
        api_ok = current[api_ok_col].fillna(False).astype(str).str.casefold().isin({"true", "1", "yes"})
        success = percent(api_ok.sum(), len(api_ok))
        prediction_health["production_api_success_rate_pct"] = check(
            success, low_is_bad(success, t.api_success_warn_pct, t.api_success_fail_pct),
            "Successful production API calls.", source=api_ok_col,
        )
    else:
        prediction_health["production_api_success_rate_pct"] = unknown("Production API call telemetry is unavailable.")
    if api_latency_col:
        latency = pd.to_numeric(current[api_latency_col], errors="coerce").dropna()
        average_latency = float(latency.mean()) if len(latency) else None
        prediction_health["production_api_average_latency_ms"] = (
            check(
                round(average_latency, 3), high_is_bad(average_latency, t.api_latency_warn_ms, t.api_latency_fail_ms),
                "Mean production API latency.", source=api_latency_col,
            ) if average_latency is not None else unknown("Production API latency is unavailable.")
        )
    else:
        prediction_health["production_api_average_latency_ms"] = unknown("Production API latency is unavailable.")

    api_sample = bundle.get("api_test_result", {}) or {}
    sample_status = str(api_sample.get("status", UNKNOWN)).upper()
    if sample_status not in {PASS, WARN, FAIL}:
        sample_status = UNKNOWN
    prediction_health["live_api_sample"] = check(
        api_sample.get("latency_ms", api_sample.get("http_status")), sample_status,
        api_sample.get("reason", "Live API sample on a current Oracle row."), source="live API smoke test",
    )

    metadata = bundle.get("model_metadata", {}) or {}
    mlflow = metadata.get("mlflow", metadata) or {}
    model_version = mlflow.get("model_version")
    deployment_health = {
        "registered_model_version": check(
            model_version, PASS if model_version not in (None, "") else UNKNOWN,
            "Resolved immutable MLflow model version.", source="MLflow registry metadata",
        ),
        "registered_model_run_id": check(
            mlflow.get("run_id"), PASS if mlflow.get("run_id") else UNKNOWN,
            "Originating MLflow run ID.", source="MLflow registry metadata",
        ),
        "model_artifact_snapshot": check(
            mlflow.get("model_inventory", {}).get("file_count"),
            PASS if mlflow.get("model_inventory", {}).get("file_count", 0) > 0 else UNKNOWN,
            "Downloaded registered-model artifact files.", source="collector artifact inventory",
        ),
    }

    telemetry = bundle.get("pipeline_telemetry", {}) or {}
    stages = telemetry.get("stages", {}) or {}
    stage_values = [str(value).upper() for value in stages.values()]
    stage_status = FAIL if FAIL in stage_values else WARN if WARN in stage_values else PASS if stage_values else UNKNOWN
    errors = int(telemetry.get("error_count", 0) or 0)
    pipeline_health = {
        "stage_statuses": check(stages or None, stage_status, "Collector and health-pipeline stage outcomes."),
        "error_count": check(errors, PASS if errors == 0 else FAIL, "Collection and scoring error count."),
        "code_version": check(
            telemetry.get("code_version"), PASS if telemetry.get("code_version") else UNKNOWN,
            "Collector build or code version.",
        ),
    }

    trusted = bool(independent.get("trusted"))
    accuracy, weighted_f1 = independent.get("accuracy"), independent.get("weighted_f1")
    independent_health = {
        "trusted_label_count": check(
            independent.get("count", 0), PASS if trusted and independent.get("count", 0) else UNKNOWN,
            "Trusted, mature post-training evaluation labels.",
        ),
        "production_accuracy": (
            check(
                round(accuracy, 6), low_is_bad(accuracy, t.accuracy_warn, t.accuracy_fail),
                "Trusted-label production accuracy.",
                threshold={"warn_below": t.accuracy_warn, "fail_below": t.accuracy_fail},
            ) if trusted and accuracy is not None else unknown("Independent trusted labels are unavailable.")
        ),
        "production_weighted_f1": (
            check(
                round(weighted_f1, 6), low_is_bad(weighted_f1, t.weighted_f1_warn, t.weighted_f1_fail),
                "Trusted-label production weighted F1.",
                threshold={"warn_below": t.weighted_f1_warn, "fail_below": t.weighted_f1_fail},
            ) if trusted and weighted_f1 is not None else unknown("Independent trusted labels are unavailable.")
        ),
    }
    challenger = independent.get("challenger", {})
    if challenger.get("available"):
        disagreement = challenger["disagreement_rate_pct"]
        independent_health["challenger_disagreement_rate_pct"] = check(
            disagreement,
            high_is_bad(disagreement, t.challenger_disagreement_warn_pct, t.challenger_disagreement_fail_pct),
            "Production disagreement with the optional TF-IDF challenger.",
        )

    sections = {
        "input_health": input_health,
        "prediction_health": prediction_health,
        "deployment_health": deployment_health,
        "pipeline_health": pipeline_health,
        "historical_comparison": comparison,
        "independent_evaluation": independent_health,
    }
    metrics = [item for section in sections.values() for item in section.values()]
    counts = {status: sum(item["status"] == status for item in metrics) for status in (PASS, WARN, FAIL, UNKNOWN)}
    failed_groups = sum(any(item["status"] == FAIL for item in section.values()) for section in sections.values())
    overall = (
        "CRITICAL" if failed_groups >= 3 else "DEGRADED" if failed_groups else
        "WARNING" if counts[WARN] else "HEALTHY" if counts[PASS] else "UNKNOWN"
    )
    collection_status = "FAILED" if n == 0 else "PARTIAL" if counts[UNKNOWN] else "COMPLETE"

    signal_map = {
        "source_not_fresh": ("input_health", "source_freshness_hours"),
        "missing_text_spike": ("input_health", "missing_text_rate_pct"),
        "short_text_spike": ("input_health", "short_text_rate_pct"),
        "preprocessing_failure": ("input_health", "preprocessing_failure_rate_pct"),
        "low_confidence_predictions": ("prediction_health", "low_confidence_rate_pct"),
        "production_api_failure": ("prediction_health", "production_api_success_rate_pct"),
        "production_api_latency": ("prediction_health", "production_api_average_latency_ms"),
        "model_accuracy_below_target": ("independent_evaluation", "production_accuracy"),
        "challenger_disagreement": ("independent_evaluation", "challenger_disagreement_rate_pct"),
        "prediction_distribution_shift": ("historical_comparison", "prediction_distribution_jsd"),
        "new_ticket_language_detected": ("historical_comparison", "unseen_token_rate_pct"),
        "text_distribution_shift": ("historical_comparison", "token_distribution_jsd"),
    }
    signals = []
    for name, (section_name, metric_name) in signal_map.items():
        item = sections.get(section_name, {}).get(metric_name)
        if item and item["status"] in {WARN, FAIL}:
            signals.append({
                "signal": name, "severity": item["status"],
                "evidence_value": item["value"], "description": item["description"],
            })

    presentation_specs = [
        ("New-data trusted accuracy", "independent_evaluation", "production_accuracy", "Oracle evaluation window with trusted labels", "Independent mature evaluation split"),
        ("New-data trusted weighted F1", "independent_evaluation", "production_weighted_f1", "Oracle evaluation window with trusted labels", "Independent mature evaluation split"),
        ("Avg prediction confidence", "prediction_health", "average_prediction_confidence", "Current Oracle window from bundle", "Collector model confidence"),
        ("Low confidence rate %", "prediction_health", "low_confidence_rate_pct", "Current Oracle window from bundle", "Collector model confidence"),
        ("Live API sample on new data", "prediction_health", "live_api_sample", "Current Oracle row sent to live API", "Live API smoke test"),
        ("New-data trusted label count", "independent_evaluation", "trusted_label_count", "Oracle evaluation window after trust filters", "Independent mature evaluation split"),
        ("MLflow model version", "deployment_health", "registered_model_version", "MLflow registry metadata", "Collector registry evidence"),
        ("Source freshness (hours)", "input_health", "source_freshness_hours", "Current Oracle window metadata", "Latest Oracle update timestamp"),
        ("Ticket count", "input_health", "ticket_count", "Current Oracle window", "Rows collected for current window"),
    ]
    key_metrics = []
    for label, section_name, metric_name, data_source, how_tested in presentation_specs:
        item = sections[section_name].get(metric_name)
        if item and item["status"] != UNKNOWN and item["value"] is not None:
            threshold = item.get("threshold")
            why = f"{item['status']}: {item['description']}"
            if threshold:
                why += f" Thresholds: {threshold}."
            key_metrics.append({
                "tested_metric": label, "value": item["value"], "status": item["status"],
                "why": why, "data_source": data_source,
                "tested_on": section_name.replace("_", " "), "how_tested": how_tested,
            })

    return json_safe({
        "overall_health_status": overall,
        "collection_status": collection_status,
        "health_summary": {
            "overall_health": overall, "collection_status": collection_status,
            "available_tests": counts[PASS] + counts[WARN] + counts[FAIL],
            "PASS": counts[PASS], "WARN": counts[WARN], "FAIL": counts[FAIL],
        },
        "detected_signals": signals,
        "key_metrics": key_metrics,
        "sections": sections,
        "independent_evaluation_evidence": independent,
        "model_metadata": metadata,
    })


def run_health_check(
    bundle_or_directory: Mapping[str, Any] | str | Path,
    config: Mapping[str, Any] | HealthConfig | None = None,
    thresholds: Mapping[str, Any] | Thresholds | None = None,
) -> dict[str, Any]:
    bundle = (
        load_collection(bundle_or_directory)
        if isinstance(bundle_or_directory, (str, Path)) else bundle_or_directory
    )
    return evaluate_health(bundle, config=config, thresholds=thresholds)


## 4. Configure

Set the collector bundle directory, optional threshold overrides, and the run flag.
Use `challenger="tfidf"` only when the lightweight challenger is needed.


In [ ]:
COLLECTION_DIRECTORY = "C:/temp/detect_ewoc/ewoc_collection_v53"
HEALTH_SETTINGS = {
    "challenger": "none",
    "label_maturity_hours": 168,
    "evaluation_max_rows": 5_000,
    "labels_independent_of_model": True,
    "output": "C:/temp/detect_ewoc/ewoc_agent_evidence.json",
}
THRESHOLD_OVERRIDES = {
    "accuracy_warn": 0.70,
    "accuracy_fail": 0.65,
    "weighted_f1_warn": 0.70,
    "weighted_f1_fail": 0.55,
}
RUN_HEALTH_NOW = True

display(pd.DataFrame({
    "setting": ["collection_directory", *HEALTH_SETTINGS, *THRESHOLD_OVERRIDES],
    "value": [COLLECTION_DIRECTORY, *HEALTH_SETTINGS.values(), *THRESHOLD_OVERRIDES.values()],
}))


## 5. Run and inspect

The output is intentionally limited to the health summary, detected signals,
and presentation metrics. The complete metric dictionary is still written to JSON.


In [ ]:
STATUS_COLORS = {
    "PASS": "background-color: #c6e6c9; font-weight: bold",
    "WARN": "background-color: #fff4c4; font-weight: bold",
    "FAIL": "background-color: #ffccd2; font-weight: bold",
    "HEALTHY": "background-color: #c6e6c9; font-weight: bold",
    "WARNING": "background-color: #fff4c4; font-weight: bold",
    "DEGRADED": "background-color: #ffddb8; font-weight: bold",
    "CRITICAL": "background-color: #ffccd2; font-weight: bold",
}

if RUN_HEALTH_NOW:
    health_result = run_health_check(
        COLLECTION_DIRECTORY,
        config=HEALTH_SETTINGS,
        thresholds=THRESHOLD_OVERRIDES,
    )
    health_summary = pd.DataFrame([health_result["health_summary"]])
    display(
        health_summary.style
        .map(lambda value: STATUS_COLORS.get(str(value), ""), subset=["overall_health"])
        .set_caption("Health summary")
    )

    signal_frame = pd.DataFrame(health_result["detected_signals"])
    if signal_frame.empty:
        print("No warning or failure signals detected.")
    else:
        display(
            signal_frame.style
            .map(lambda value: STATUS_COLORS.get(str(value), ""), subset=["severity"])
            .set_caption("Detected signals")
        )

    key_frame = pd.DataFrame(health_result["key_metrics"])
    display(
        key_frame.style
        .map(lambda value: STATUS_COLORS.get(str(value), ""), subset=["status"])
        .set_caption("Presentation summary: what was tested and how")
    )

    output_path = Path(HEALTH_SETTINGS["output"])
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text(json.dumps(health_result, indent=2, ensure_ascii=False), encoding="utf-8")
    print(f"Evidence written to: {output_path}")
else:
    print("Set RUN_HEALTH_NOW=True, rerun the configuration cell, then run this cell.")
